# HCRG Gate Probing — Prediction P4

Loads trained HCRG checkpoints and analyzes gate activation patterns to determine if gates learned **selective, context-dependent suppression** (paper's P4) or collapsed to uniform dampening.

Key questions:
1. **Distribution**: Are gate values spread out, or clustered near 1.0 (never learned)?
2. **Sparsity**: What fraction of gates actively suppress heads (< 0.5)?
3. **Context dependence**: Does the same head get different gate values on different inputs?
4. **Layer structure**: Do deeper layers show more gating activity?

In [ ]:
import sys, os
import torch
import numpy as np
import tiktoken
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (12, 5)

sys.path.insert(0, os.path.abspath('.'))
from custom_model import GPTConfig, GPT

DEVICE = 'cpu'
enc = tiktoken.get_encoding('gpt2')
print(f'torch {torch.__version__}, device={DEVICE}')

In [ ]:
def load_hcrg_checkpoint(path, device='cpu'):
    """Load an HCRG checkpoint and return (model, config)."""
    ckpt = torch.load(path, map_location=device)
    model_args = ckpt['model_args']
    conf = GPTConfig(**model_args)
    model = GPT(conf)

    state_dict = ckpt['model']
    # strip torch.compile prefix if present
    for k in list(state_dict.keys()):
        if k.startswith('_orig_mod.'):
            state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)
    return model, ckpt['config']

In [ ]:
def collect_gates(model, token_ids, device='cpu'):
    """
    Run a forward pass and capture gate activations from every HCRGBlock.
    Returns dict: layer_idx -> tensor of shape (T, n_head) with sigmoid values.
    """
    gate_activations = {}
    hooks = []

    for i, block in enumerate(model.transformer.h):
        def make_hook(layer_idx):
            def hook_fn(module, inp, out):
                gate_activations[layer_idx] = torch.sigmoid(out).detach().cpu()
            return hook_fn
        h = block.gate_proj.register_forward_hook(make_hook(i))
        hooks.append(h)

    idx = torch.tensor([token_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        model(idx)

    for h in hooks:
        h.remove()

    # squeeze batch dim: (1, T, n_head) -> (T, n_head)
    return {k: v.squeeze(0) for k, v in gate_activations.items()}


def encode_text(text, max_len=512):
    tokens = enc.encode(text)
    return tokens[:max_len]

In [ ]:
PROMPTS = {
    'story': (
        "Once upon a time, there was a little girl named Lily. She loved to play "
        "in the garden with her dog Max. One sunny day, they found a beautiful "
        "butterfly sitting on a flower. Lily wanted to catch it, but Max barked "
        "and the butterfly flew away. Lily was sad, but then she saw a rainbow "
        "in the sky and smiled. She knew that tomorrow would be another adventure."
    ),
    'factual': (
        "The mitochondria is the powerhouse of the cell. It produces adenosine "
        "triphosphate through oxidative phosphorylation. The process involves "
        "the electron transport chain located in the inner mitochondrial membrane. "
        "Glucose is first broken down through glycolysis in the cytoplasm, producing "
        "pyruvate which enters the mitochondrial matrix for the citric acid cycle."
    ),
    'code': (
        "def fibonacci(n):\n    if n <= 1:\n        return n\n    "
        "a, b = 0, 1\n    for _ in range(2, n + 1):\n        "
        "a, b = b, a + b\n    return b\n\n"
        "print([fibonacci(i) for i in range(20)])"
    ),
    'repetitive': 'the cat sat on the mat. ' * 20,
}

## Micro Model (4 layers, 4 heads, 256 embed)

In [ ]:
micro_model, micro_cfg = load_hcrg_checkpoint(
    'out-run3/out/micro/hcrg/seed42/ckpt.pt', device=DEVICE
)
print(f"Micro model: {micro_cfg.get('n_layer')}L, {micro_cfg.get('n_head')}H, {micro_cfg.get('n_embd')}E")

In [ ]:
# Collect gates for all prompts
micro_gates = {}
for name, text in PROMPTS.items():
    tokens = encode_text(text)
    micro_gates[name] = collect_gates(micro_model, tokens, device=DEVICE)
    print(f'{name}: {len(tokens)} tokens, gates shape per layer: {micro_gates[name][0].shape}')

In [ ]:
def plot_gate_distribution(all_gates, title_prefix=''):
    """Histogram of all gate values across all layers, heads, and positions."""
    all_vals = []
    for prompt_gates in all_gates.values():
        for layer_gates in prompt_gates.values():
            all_vals.append(layer_gates.numpy().flatten())
    all_vals = np.concatenate(all_vals)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(all_vals, bins=100, edgecolor='none', alpha=0.8)
    axes[0].set_xlabel('Gate value (sigmoid output)')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{title_prefix}Gate Distribution (all prompts)')
    axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='0.5 threshold')
    axes[0].legend()

    # Zoomed view on lower range
    axes[1].hist(all_vals[all_vals < 0.95], bins=100, edgecolor='none', alpha=0.8, color='orange')
    axes[1].set_xlabel('Gate value')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'{title_prefix}Zoomed: gates < 0.95')

    below_05 = (all_vals < 0.5).mean() * 100
    below_09 = (all_vals < 0.9).mean() * 100
    below_095 = (all_vals < 0.95).mean() * 100
    print(f'Gate statistics ({title_prefix.strip()}):')
    print(f'  Mean: {all_vals.mean():.4f}, Std: {all_vals.std():.4f}')
    print(f'  Min: {all_vals.min():.4f}, Max: {all_vals.max():.4f}')
    print(f'  < 0.5 (actively suppressing): {below_05:.2f}%')
    print(f'  < 0.9: {below_09:.2f}%')
    print(f'  < 0.95: {below_095:.2f}%')

    plt.tight_layout()
    plt.show()

plot_gate_distribution(micro_gates, title_prefix='Micro — ')

In [ ]:
def plot_per_head_stats(gates_dict, prompt_name, n_layer, n_head, title_prefix=''):
    """Heatmaps of mean and std gate value per (layer, head) for a single prompt."""
    gates = gates_dict[prompt_name]
    means = np.zeros((n_layer, n_head))
    stds = np.zeros((n_layer, n_head))

    for layer in range(n_layer):
        g = gates[layer].numpy()  # (T, n_head)
        means[layer] = g.mean(axis=0)
        stds[layer] = g.std(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, max(3, n_layer * 0.6)))

    im0 = axes[0].imshow(means, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    axes[0].set_xlabel('Head')
    axes[0].set_ylabel('Layer')
    axes[0].set_title(f'{title_prefix}Mean gate — "{prompt_name}"')
    axes[0].set_xticks(range(n_head))
    axes[0].set_yticks(range(n_layer))
    plt.colorbar(im0, ax=axes[0], shrink=0.8)

    im1 = axes[1].imshow(stds, cmap='hot', vmin=0, aspect='auto')
    axes[1].set_xlabel('Head')
    axes[1].set_ylabel('Layer')
    axes[1].set_title(f'{title_prefix}Std gate (context variation) — "{prompt_name}"')
    axes[1].set_xticks(range(n_head))
    axes[1].set_yticks(range(n_layer))
    plt.colorbar(im1, ax=axes[1], shrink=0.8)

    for ax in axes:
        for i in range(n_layer):
            for j in range(n_head):
                val = means[i, j] if ax == axes[0] else stds[i, j]
                ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=7,
                        color='black' if val > 0.4 else 'white')

    plt.tight_layout()
    plt.show()

for pname in PROMPTS:
    plot_per_head_stats(micro_gates, pname, n_layer=4, n_head=4, title_prefix='Micro — ')

In [ ]:
def plot_cross_prompt_comparison(all_gates, n_layer, n_head, title_prefix=''):
    """Compare mean gate patterns across prompts to test context dependence."""
    prompt_names = list(all_gates.keys())
    n_prompts = len(prompt_names)

    fig, axes = plt.subplots(1, n_prompts, figsize=(4 * n_prompts, max(3, n_layer * 0.6)))
    if n_prompts == 1:
        axes = [axes]

    all_means = []
    for idx, pname in enumerate(prompt_names):
        gates = all_gates[pname]
        means = np.zeros((n_layer, n_head))
        for layer in range(n_layer):
            means[layer] = gates[layer].numpy().mean(axis=0)
        all_means.append(means)

        im = axes[idx].imshow(means, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        axes[idx].set_title(pname, fontsize=10)
        axes[idx].set_xlabel('Head')
        if idx == 0:
            axes[idx].set_ylabel('Layer')
        axes[idx].set_xticks(range(n_head))
        axes[idx].set_yticks(range(n_layer))
        for i in range(n_layer):
            for j in range(n_head):
                axes[idx].text(j, i, f'{means[i,j]:.2f}', ha='center', va='center',
                               fontsize=7)

    plt.suptitle(f'{title_prefix}Mean gate per (layer, head) across prompts', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

    # Quantify cross-prompt variation
    all_means = np.stack(all_means)  # (n_prompts, n_layer, n_head)
    cross_prompt_std = all_means.std(axis=0)  # (n_layer, n_head)
    print(f'\nCross-prompt std of mean gate (higher = more context-dependent):')
    print(f'  Overall mean cross-prompt std: {cross_prompt_std.mean():.4f}')
    print(f'  Max cross-prompt std: {cross_prompt_std.max():.4f} '
          f'at layer {np.unravel_index(cross_prompt_std.argmax(), cross_prompt_std.shape)[0]}, '
          f'head {np.unravel_index(cross_prompt_std.argmax(), cross_prompt_std.shape)[1]}')

plot_cross_prompt_comparison(micro_gates, n_layer=4, n_head=4, title_prefix='Micro — ')

In [ ]:
def plot_position_gates(gates_dict, prompt_name, n_layer, n_head, title_prefix=''):
    """Show gate values across token positions for each head, one subplot per layer."""
    gates = gates_dict[prompt_name]
    fig, axes = plt.subplots(n_layer, 1, figsize=(14, 2.5 * n_layer), sharex=True)
    if n_layer == 1:
        axes = [axes]

    for layer in range(n_layer):
        g = gates[layer].numpy()  # (T, n_head)
        for head in range(n_head):
            axes[layer].plot(g[:, head], label=f'H{head}', alpha=0.8, linewidth=0.8)
        axes[layer].set_ylabel(f'L{layer} gate')
        axes[layer].set_ylim(-0.05, 1.05)
        axes[layer].legend(loc='upper right', fontsize=7, ncol=n_head)
        axes[layer].axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)

    axes[-1].set_xlabel('Token position')
    plt.suptitle(f'{title_prefix}Gate values by position — "{prompt_name}"', fontsize=11)
    plt.tight_layout()
    plt.show()

plot_position_gates(micro_gates, 'story', n_layer=4, n_head=4, title_prefix='Micro — ')
plot_position_gates(micro_gates, 'factual', n_layer=4, n_head=4, title_prefix='Micro — ')

In [ ]:
def gate_weight_analysis(model, title_prefix=''):
    """Analyze the learned gate_proj weights and biases directly."""
    print(f'{title_prefix}Gate projection parameters:')
    print(f'{"Layer":<8} {"Bias (raw)":<40} {"Bias (sigmoid)":<40} {"W norm":<10}')
    print('-' * 100)
    for i, block in enumerate(model.transformer.h):
        gp = block.gate_proj
        bias = gp.bias.data.cpu().numpy()
        sig_bias = 1.0 / (1.0 + np.exp(-bias))
        w_norm = gp.weight.data.cpu().norm().item()
        bias_str = ', '.join(f'{b:+.3f}' for b in bias)
        sig_str = ', '.join(f'{s:.3f}' for s in sig_bias)
        print(f'L{i:<7} [{bias_str}]  [{sig_str}]  {w_norm:.4f}')

    # How far did biases move from init (-5.0)?
    all_biases = []
    for block in model.transformer.h:
        all_biases.extend(block.gate_proj.bias.data.cpu().numpy().tolist())
    all_biases = np.array(all_biases)
    print(f'\nBias drift from init (+5.0):')
    print(f'  Mean bias: {all_biases.mean():.3f} (init: +5.000, delta: {all_biases.mean() - 5:.3f})')
    print(f'  Std bias: {all_biases.std():.3f}')
    print(f'  Range: [{all_biases.min():.3f}, {all_biases.max():.3f}]')

gate_weight_analysis(micro_model, title_prefix='Micro — ')

## Standard Model (12 layers, 12 heads, 768 embed)

In [ ]:
std_model, std_cfg = load_hcrg_checkpoint(
    'out-run3/out/standard/hcrg/seed42/ckpt.pt', device=DEVICE
)
print(f"Standard model: {std_cfg.get('n_layer')}L, {std_cfg.get('n_head')}H, {std_cfg.get('n_embd')}E")

In [ ]:
std_gates = {}
for name, text in PROMPTS.items():
    tokens = encode_text(text)
    std_gates[name] = collect_gates(std_model, tokens, device=DEVICE)
    print(f'{name}: {len(tokens)} tokens')

In [ ]:
plot_gate_distribution(std_gates, title_prefix='Standard — ')

In [ ]:
for pname in ['story', 'factual']:
    plot_per_head_stats(std_gates, pname, n_layer=12, n_head=12, title_prefix='Standard — ')

In [ ]:
plot_cross_prompt_comparison(std_gates, n_layer=12, n_head=12, title_prefix='Standard — ')

In [ ]:
# Position-wise gates for selected layers (first, middle, last)
for layer_set_name, layers in [('early', [0, 1, 2]), ('mid', [5, 6, 7]), ('late', [9, 10, 11])]:
    gates = std_gates['story']
    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    for ax_idx, layer in enumerate(layers):
        g = gates[layer].numpy()
        for head in range(12):
            axes[ax_idx].plot(g[:, head], alpha=0.6, linewidth=0.6)
        axes[ax_idx].set_ylabel(f'L{layer}')
        axes[ax_idx].set_ylim(-0.05, 1.05)
        axes[ax_idx].axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
    axes[-1].set_xlabel('Token position')
    plt.suptitle(f'Standard — {layer_set_name} layers — "story"', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
gate_weight_analysis(std_model, title_prefix='Standard — ')

## Summary & Comparison

In [ ]:
def summarize_gates(all_gates, n_layer, n_head, label):
    """Produce a summary table of gate behavior."""
    all_vals = []
    per_prompt_means = []
    for pname, gates in all_gates.items():
        means = np.zeros((n_layer, n_head))
        for layer in range(n_layer):
            g = gates[layer].numpy()
            all_vals.append(g.flatten())
            means[layer] = g.mean(axis=0)
        per_prompt_means.append(means)

    all_vals = np.concatenate(all_vals)
    stacked = np.stack(per_prompt_means)  # (n_prompts, n_layer, n_head)

    within_prompt_std = []
    for pname, gates in all_gates.items():
        for layer in range(n_layer):
            within_prompt_std.append(gates[layer].numpy().std(axis=0).mean())

    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'  Overall mean gate value:      {all_vals.mean():.4f}')
    print(f'  Overall std:                  {all_vals.std():.4f}')
    print(f'  Fraction < 0.5 (suppressing): {(all_vals < 0.5).mean()*100:.2f}%')
    print(f'  Fraction < 0.9:               {(all_vals < 0.9).mean()*100:.2f}%')
    print(f'  Fraction > 0.99 (fully open): {(all_vals > 0.99).mean()*100:.2f}%')
    print(f'  Within-prompt std (mean):     {np.mean(within_prompt_std):.4f}')
    print(f'  Cross-prompt std (mean):      {stacked.std(axis=0).mean():.4f}')

    verdict = []
    if all_vals.mean() > 0.99:
        verdict.append('Gates near-fully open — minimal learning')
    elif all_vals.mean() > 0.95:
        verdict.append('Gates mostly open — modest suppression learned')
    else:
        verdict.append('Gates show substantial suppression activity')

    if np.mean(within_prompt_std) > 0.05:
        verdict.append('Context-dependent (within-prompt variation detected)')
    else:
        verdict.append('Low within-prompt variation — gates may be position/content insensitive')

    if stacked.std(axis=0).mean() > 0.02:
        verdict.append('Input-dependent (cross-prompt variation detected)')
    else:
        verdict.append('Low cross-prompt variation — gates may have collapsed to fixed values')

    print(f'\n  Verdict:')
    for v in verdict:
        print(f'    • {v}')

summarize_gates(micro_gates, n_layer=4, n_head=4, label='MICRO (4L/4H/256E) — TinyStories')
summarize_gates(std_gates, n_layer=12, n_head=12, label='STANDARD (12L/12H/768E) — FineWeb-Edu')